# 29 — Temperature, sampling, and why low is boring

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/029-alignment-and-inference.ipynb)

A language model ends every step with one score per candidate word. Temperature divides those scores before the softmax: low values make the popular word almost certain, high values make the choice nearly uniform. Here you turn that one dial and watch the distribution, the sampled text, and the failure at the bottom of the range.

## 1. The numbers

Three candidate words with raw scores 2, 1 and 0. The softmax turns scores into probabilities, and dividing the scores by a temperature first changes how sharply the largest score dominates.

In [ ]:
import numpy as np                                     # the array library, already on Colab

logits = np.array([2., 1., 0.])                       # one score per candidate word
names = ["cat", "dog", "hat"]                         # what the three candidates are

def softmax(z):                                       # scores in, probabilities out
    e = np.exp(z - z.max())                           # subtract the max, then exponentiate
    return e / e.sum()                                # share of the total

for T in [0.5, 1.0, 2.0]:                             # three temperatures to compare
    p = softmax(logits / T)                           # divide the scores, then softmax
    print(f"T={T:3.1f}  probs={np.round(p, 3)}  top={p.max():.3f}")   # one line per temperature

print("entropy at T=1:", round(float(-(softmax(logits) * np.log(softmax(logits))).sum()), 3))   # nats

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import numpy as np</code> — load NumPy for the arithmetic.</li><li><code>logits = np.array([2., 1., 0.])</code> — the three raw scores the model produced.</li><li><code>names = [...]</code> — labels, so the numbers have something to point at.</li><li><code>def softmax(z)</code> — score vector in, probability vector out.</li><li><code>e = np.exp(z - z.max())</code> — subtract the largest score first; the ratios do not change.</li><li><code>return e / e.sum()</code> — divide by the total so the probabilities add to one.</li><li><code>for T in [0.5, 1.0, 2.0]</code> — three settings of the dial.</li><li><code>p = softmax(logits / T)</code> — dividing by <code>T</code> before the softmax is the whole trick.</li><li><code>print(f"...")</code> — <code>:3.1f</code> and <code>:.3f</code> fix the number of decimals.</li><li><code>-(p * np.log(p)).sum()</code> — entropy: how spread out the distribution is.</li></ul>

</details>

## 2. Step by step: one temperature at a time

Drop the temperature from 2.0 to 0.1 and print the scaled scores, the exponentials and the probabilities at each stop. The exponentials are where the gap between first and second place gets multiplied.

In [ ]:
for T in [2.0, 1.0, 0.5, 0.2, 0.1]:                   # from flat to sharp
    z = logits / T                                    # the scored, divided by the temperature
    e = np.exp(z - z.max())                           # exponentials, max removed
    p = e / e.sum()                                   # probabilities
    gap = z[0] - z[1]                                 # distance between best and runner-up
    print(f"T={T:4.1f} scaled={np.round(z, 2)} gap={gap:6.1f} top={p[0]:.4f}")   # inspect the step
    print("        p =", np.round(p, 4))              # the full distribution on its own line

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>for T in [2.0, 1.0, 0.5, 0.2, 0.1]</code> — a slide from flat to sharp.</li><li><code>z = logits / T</code> — every score divided by the same number keeps their order.</li><li><code>e = np.exp(z - z.max())</code> — exponentiate; the result is always positive.</li><li><code>p = e / e.sum()</code> — normalise into probabilities.</li><li><code>gap = z[0] - z[1]</code> — the margin between the top two, after scaling.</li><li><code>print(f"...")</code> — the scaled scores, the margin and the top probability.</li><li><code>print("p =", ...)</code> — the whole distribution, so you can see the tail lose its mass.</li></ul>

</details>

## 3. Plot: the distribution at three temperatures

The same three candidates under three settings of the dial. At 0.5 the winner takes most of the mass; at 2.0 the three bars are almost the same height.

In [ ]:
import matplotlib.pyplot as plt                       # plotting library, already on Colab

fig, axes = plt.subplots(1, 3, figsize=(9, 3), sharey=True)   # three panels, one shared y-axis
for ax, T in zip(axes, [0.5, 1.0, 2.0]):              # one panel per temperature
    p = softmax(logits / T)                           # probabilities at this temperature
    ax.bar(names, p, color="tab:blue")                # one bar per candidate
    ax.set_ylim(0, 1)                                 # the same scale in every panel
    ax.set_xlabel("candidate word")                   # x-axis label
    ax.set_title(f"T = {T}")                          # panel title
axes[0].set_ylabel("probability")                     # one y-axis label is enough
plt.suptitle("low T picks a favourite; high T keeps the options open")   # the takeaway
plt.tight_layout()                                    # stop the labels overlapping
plt.show()                                            # render it

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import matplotlib.pyplot as plt</code> — load the plotting library.</li><li><code>fig, axes = plt.subplots(1, 3, ...)</code> — a row of three panels sharing the y-axis.</li><li><code>for ax, T in zip(axes, [0.5, 1.0, 2.0])</code> — one temperature per panel.</li><li><code>p = softmax(logits / T)</code> — that temperature's distribution.</li><li><code>ax.bar(names, p)</code> — draw one bar per candidate word.</li><li><code>ax.set_ylim(0, 1)</code> — fix the scale so the panels are comparable.</li><li><code>axes[0].set_ylabel("probability")</code> — label the shared y-axis once.</li><li><code>plt.suptitle(...)</code> — the takeaway for the whole figure.</li></ul>

</details>

## 4. Interactive: turn the dial

Drag the temperature and watch the three bars move. The title reports the top probability and the entropy, so you can see one number rise exactly as the other falls.

In [ ]:
from ipywidgets import interact                      # sliders that rerun a function

def show(T=1.0):                                     # T comes from the slider
    p = softmax(logits / T)                          # probabilities at this temperature
    ent = float(-(p * np.log(p)).sum())              # entropy of the distribution
    plt.figure(figsize=(4, 3))                       # a fresh figure on every drag
    plt.bar(names, p, color="tab:blue")              # the three probabilities
    plt.ylim(0, 1)                                   # fixed axis, so heights are comparable
    plt.xlabel("candidate word")                     # x-axis label
    plt.ylabel("probability")                        # y-axis label
    plt.title(f"T={T:.2f}  top={p.max():.3f}  entropy={ent:.2f}")   # live readout
    plt.show()                                       # draw the frame

interact(show, T=(0.1, 3.0, 0.1))                    # attach the slider

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>from ipywidgets import interact</code> — Colab ships the widget library.</li><li><code>def show(T=1.0)</code> — the function that draws one frame.</li><li><code>p = softmax(logits / T)</code> — the distribution at the slider's temperature.</li><li><code>ent = ...</code> — entropy in nats; high means spread out.</li><li><code>plt.bar(names, p)</code> — the three probabilities as bars.</li><li><code>plt.ylim(0, 1)</code> — keep the axis fixed while dragging.</li><li><code>plt.title(...)</code> — the live readout of temperature, top probability and entropy.</li><li><code>interact(show, T=(0.1, 3.0, 0.1))</code> — the slider, its range and step.</li></ul>

</details>

## 5. When it breaks: T near zero repeats itself

As the temperature approaches zero the scaled gap between first and second place grows without limit, and the top probability approaches one. Sampling then stops being sampling: every draw is the same word.

In [ ]:
Ts = np.linspace(0.05, 2.0, 60)                     # sixty temperatures, from tiny upwards
tops = np.array([softmax(logits / T)[0] for T in Ts])   # the top probability at each one

p_cold = softmax(logits / 0.15)                       # the distribution at T = 0.15
rng = np.random.default_rng(0)                        # a fixed generator, so draws repeat
draws = rng.choice(3, size=20, p=p_cold)              # twenty sampled words at T = 0.15

print("T=0.15 gives p =", np.round(p_cold, 4))       # the top bar is 0.999
print("twenty draws   =", draws)                      # all zeros: 'cat' every time
print("distinct words =", len(set(draws.tolist())))   # one

plt.figure(figsize=(6, 3))                            # a wide, short figure
plt.plot(Ts, tops, color="tab:red")                   # top probability against temperature
plt.axhline(0.999, color="black", linestyle="--")     # the "always the same word" level
plt.axvline(0.15, color="grey", linestyle=":")        # where we sampled above
plt.xlabel("temperature")                             # x-axis label
plt.ylabel("probability of the top word")             # y-axis label
plt.title("below T = 0.2 the top word wins 999 times in 1000")   # the takeaway
plt.show()                                            # render it

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>Ts = np.linspace(0.05, 2.0, 60)</code> — a grid of temperatures for the curve.</li><li><code>tops = [...]</code> — a list comprehension: the top probability at each temperature.</li><li><code>p_cold = softmax(logits / 0.15)</code> — the distribution at a very low temperature.</li><li><code>rng = np.random.default_rng(0)</code> — seeded, so the draws are reproducible.</li><li><code>rng.choice(3, size=20, p=p_cold)</code> — sample twenty words from those probabilities.</li><li><code>print(...)</code> — the probabilities, the draws, and how many distinct words appeared.</li><li><code>plt.plot(Ts, tops)</code> — top probability against temperature.</li><li><code>plt.axhline(0.999, ...)</code> and <code>plt.axvline(0.15, ...)</code> — mark the failure level and the sample point.</li></ul>

</details>

## Your turn

Change the logits to `[3., 2., 0.]` and rerun section 5: the curve shifts left, so a higher temperature is now needed to keep the options open. Find the temperature at which the top probability first drops below 0.9, and then check that section 4's slider agrees with you.